In [1]:
import sklearn as sk
from sklearn import set_config
set_config(display='text')

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, Activation, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.utils import to_categorical

tf.random.set_seed(1)

In [2]:
import platform
import numpy as np
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.font_manager as fm
import warnings
warnings.filterwarnings('ignore')


# OS에 따라서 폰트 로드
os_name = platform.system()

if os_name == 'Linux':
    # 리눅스용 폰트 로드
    plt.rc('font', family='NanumGothicCoding')
    print("(리눅스) 한글폰트 로드 완료")

elif os_name == 'Windows':
    # 윈도우용 폰트 로드
    font_path = 'C:\\Windows\\Fonts\\malgun.ttf'
    font = fm.FontProperties(fname=font_path).get_name()
    mpl.rc('font', family=font)
    print("(윈도우) 한글폰트 로드 완료")

elif os_name == 'Darwin':
    # 맥용 폰트 로드
    plt.rc('font', family='AppleGothic')
    print("(맥) 한글폰트 로드 완료")

else:
    print(f"알 수 없는 OS입니다: {os_name}")

# 마이너스 기호 깨짐 방지
mpl.rcParams['axes.unicode_minus'] = False

(윈도우) 한글폰트 로드 완료


In [ ]:
import pandas as pd
data = pd.read_csv('./AICE Associate 샘플문항_분류/wine_quality_data.csv')
data.head(4)

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(15,6))
sns.countplot(data=data, x="E_FE_wine_kind", ax=axes[0])
sns.histplot(data=data, x="FE_points_winery", hue='Grade', ax=axes[1])

plt.setp(axes[0].get_xticklabels(), rotation=45, ha='right')

axes[0].set_title('Wine Count')
axes[1].set_title('Winery Score')

In [ ]:
sns.boxplot(data=data, x='Grade', y='FE_points_winery')


In [ ]:
q1 = data['FE_points_winery'].quantile(0.25)
q3 = data['FE_points_winery'].quantile(0.75)
iqr = q3 - q1

lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr

data_temp = data.drop(data[(data['FE_points_winery'] > upper_fence)|(data['FE_points_winery'] < lower_fence)].index, axis=0)
data_temp = data_temp.drop(columns=['E_title'])
data_temp.reset_index(drop=True, inplace=True)


In [ ]:
print(data_temp.isnull().sum())
data_na = data_temp.dropna()
print('----------------------\n',data_na.isnull().sum())

In [ ]:
data_na.loc[data_na['E_FE_wine_kind'].isin(['white|rose|sparkling','red|white|sparkling', 'red|white|rose|sparkling', 'red|sparkling']),['E_FE_wine_kind']] = 'sparkling'
data_na.loc[data_na['E_FE_wine_kind'].isin(['white|sparkling']),['E_FE_wine_kind']] = 'white'
data_na.loc[data_na['E_FE_wine_kind'].isin(['red|rose', 'red|white', 'red|rose|sparkling']),['E_FE_wine_kind']] = 'red'

print(data_na['E_FE_wine_kind'].value_counts())

data_na.drop(columns=['E_province', 'E_region_1', 'E_region_2', 'E_winery', 'E_wine_variety'], inplace=True)
data_preset = pd.get_dummies(data=data_na, columns=['E_country', 'E_FE_wine_kind'])
data_preset.info()

In [ ]:
from sklearn.model_selection import train_test_split

X = data_preset.drop('Grade',axis=1)
y = data_preset['Grade']

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size = 0.3, random_state=7, stratify=y)


In [ ]:
from sklearn.preprocessing import StandardScaler

ss=StandardScaler()

X_train = ss.fit_transform(X_train)
X_valid = ss.transform(X_valid)

print(round(np.max(X_valid)))

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV

dt = DecisionTreeClassifier(random_state=7)

param_grid_dt = {
    'max_depth' : [3,5,7,10],
    'min_samples_split' : [2,3,5,10],
}

gs_dt = GridSearchCV(
    estimator=dt,
    param_grid=param_grid_dt,
    cv =3 ,
    scoring='accuracy',
    n_jobs=-1,
)

gs_dt.fit(X_train, y_train)


In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(random_state=7)

param_grid_rf = {
    'max_depth' : [5,10,20],
    'min_samples_split': [2,5,10],
    'n_estimators' : [100,200,500],
}

gs_rf = GridSearchCV(
    estimator=rf,
    param_grid=param_grid_rf,
    cv=3,
    scoring='accuracy',
    n_jobs=-1,
)

gs_rf.fit(X_train, y_train)

In [ ]:
print(gs_rf.best_estimator_.n_estimators)

In [ ]:
fi = pd.DataFrame({
    'feature' : X.columns,
    'importance' : gs_rf.best_estimator_.feature_importances_,
})

fi.sort_values(['importance'],ascending=False)[:10]

sns.barplot(data=fi, x='importance', y='feature', palette='viridis')

In [ ]:
from sklearn.metrics import accuracy_score

y_pred_dt = gs_dt.best_estimator_.predict(X_valid)
y_pred_rf = gs_rf.best_estimator_.predict(X_valid)

dt_acc = accuracy_score(y_valid, y_pred_dt)
rf_acc = accuracy_score(y_valid, y_pred_rf)

print("DecisionTree_acc = ", dt_acc)
print("RandomForest_acc = ", rf_acc)

In [ ]:
from sklearn.preprocessing import LabelEncoder

model = Sequential([
    Dense(128, activation='relu', input_dim = X_train.shape[1]),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(2, activation='sigmoid'),
])

estop = EarlyStopping(monitor='val_loss', patience=14)
model.compile(
    optimizer='adam',
    loss = 'binary_crossentropy',
    metrics=['accuracy']
)

le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_valid = le.transform(y_valid)

y_train = to_categorical(y_train)
y_valid = to_categorical(y_valid)

history = model.fit(X_train, y_train,
                    epochs=30,
                    batch_size=128,
                    validation_data = (X_valid, y_valid),
                    callbacks=[estop])

In [ ]:
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend(['acc', 'val_acc'])
plt.show()